In [ ]:
from tapas_gmm.dataset.scene import SceneDataset
from pathlib import Path
from tapas_gmm.dataset.bc import BCDataset, BCDataConfig
from torch.utils.data import DataLoader
from tapas_gmm.utils.observation import collate
from tapas_gmm.dataset.demos import Demos

from tapas_gmm.policy.models.tpgmm import (
    AutoTPGMM,
    AutoTPGMMConfig,
    TPGMMConfig,
    FrameSelectionConfig,
    DemoSegmentationConfig,
    InitStrategy,
    FittingStage,
)

In [ ]:
data_root = Path("../outputs/bimanual_dataset")

In [ ]:
loaded_dataset = SceneDataset(
    data_root=Path(data_root)
)

In [ ]:
bc_config = BCDataConfig(
    fragment_length = -1,
    cameras = tuple(),
)

bc_dataset = BCDataset(
    scene_dataset=loaded_dataset,
    config=bc_config,
)

In [ ]:
loader = DataLoader(
    bc_dataset,
    collate_fn=collate,
)

In [ ]:
trajs = []

for traj in loader:
    trajs.append(traj[0])

In [ ]:
demos = Demos(trajectories = trajs)


In [ ]:
tpgmm_config = TPGMMConfig(
    add_time_component = True,
    position_only = True,
    add_action_component = False,
)

segmentation_config = DemoSegmentationConfig(
    distance_based = False,
    velocity_based = True,
)

auto_config = AutoTPGMMConfig(
    tpgmm=tpgmm_config,
    demos_segmentation=segmentation_config,
)


In [ ]:
model = AutoTPGMM(auto_config)

In [ ]:
lik, avg_loglik = model.fit_trajectories(
    demos = demos,
    fix_frames=False,
    fitting_actions = [ 
        FittingStage.INIT,
        FittingStage.EM_HMM,
    ],
)

In [ ]:
model.plot_model(
    rotations_raw=False,
)